In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install qutip

import numpy as np
import qutip as qt
from scipy.stats import multinomial
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 52.7 MB/s eta 0:00:00:00:0100:01


In [ ]:
d = 3 #starting with 3 dimensional Hilbert Space

N = d**2 #Number of POVM outcomes

def generate_square_root_povm(d, N):

    #generating haar random kets
    kets = [qt.rand_ket_haar(d) for i in range(N)]

    #what happens is picking a Haar-random Unitary Matriux Uin d dimensional unitary space and applying it to a fixed state like |0>

    S = sum(ket * ket.dag() for ket in kets) #taking the density matrices of the pure states

    evals, evecs= S.eigenstates() #inverse square root of S via eigen decomposition

    S_inv_sqrt = sum((1/np.sqrt(evals[k]))*evecs[k]*evecs[k].dag() for k in range(d))

    # building the POVMs

    povm = []
    for ket in kets:
        proj = ket*ket.dag()
        Pi = S_inv_sqrt*proj*S_inv_sqrt
        povm.append(Pi)
    return povm

print(f"Generating POVM for d={d} with N={N} outcomes...")

povm_qobj = generate_square_root_povm(d, N)

povm_np = [Pi.full() for Pi in povm_qojb]
    

In [ ]:
def probabilities(rho, povm_np):
    rho_mat = rho.full()
    p = [np.trace(rho_mat @ Pi).real for Pi in povm_np] #taking --> Tr(rhp.Pi)
    return np.array(p, dtype=np.float32)

def cholesky_vector(rho):
    rho_mat = rho_full()
    eigvals = np.linalg.eigvalsh(rho_mat)
    if np.min(eigvals) < 0:
        rho_mat += (-np.min(eigvals) + 1e-12)*np.eye(d)
    
    L = np.linlag.cholesky(rho_mat) #L is lower triangular, complex
    vec = []
    # Diagonal elements (real)
    for i in range(d):
        vec.append(L[i, i].real)

    # Off-diagonal elements (complex)
    for i in range(d):
        for j in range(i):
            vec.append(L[i, j].real)
            vec.append(L[i, j].imag)

    return np.array(vec, dtype=float)
            

In [ ]:
def random_density_matrix_ginibre(d):
    #creating a random density matrix rho = XX+ / Tr(XX+) using ginibre MATRIX 

    return qt.rand_dm(d, density = 1, distribution = 'ginibre')